# Automatidata: quién deja propina generosa, y si se puede saber

**Curso 5 del certificado, proyecto de modelos basados en árboles.**

El encargo es predecir quién dejará propina generosa, para que el conductor lo sepa antes de
aceptar la carrera.

**La pregunta ética va antes que el modelo, no después.** Un sistema que dice al conductor
quién va a dar propina puede acabar decidiendo **a quién se le para el taxi**. Eso es
discriminación con apariencia de eficiencia. La pregunta se hace ahora, sin saber todavía si
el modelo funciona, porque hacerla después de ver un buen resultado ya no cuenta.

**Lo que espero, escrito antes de ajustar nada:** dar propina es una decisión de la persona,
y las columnas de este archivo describen el viaje. Lo probable es que no se pueda predecir.
Si es así, el entregable es decirlo con pruebas.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "projects" / "curso5").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "projects"))
sys.path.insert(0, str(ROOT / "projects" / "curso5" / "automatidata" / "02_scripts"))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

from automatidata_trees import (CARD, DEFAULT_BUTTON, FEATURES, SEED, build_target, load,
                                split)
from common import Results, dataset

pd.set_option("display.width", 120)
print("listo")

listo


## 1. La fuga que hay que descartar antes de nada

`total_amount` podría contener la propina. Si la contiene, esa columna tiene la respuesta
dentro y no puede ser una variable predictora. Se comprueba, no se supone.

In [2]:
results = Results("automatidata", "Automatidata, cuaderno del Curso 5", "curso5")
df, card = load(results)


2. El efectivo, y por qué el proyecto se queda solo con la tarjeta

1. La fuga que hay que descartar antes de nada
  filas donde el importe total es la suma de las partes, propina incluida: 22656
  el importe total contiene la propina en 22656 de 22699 viajes
  así que total_amount NO puede ser variable predictora
  viajes en el CSV: 22699
    % con propina cero en el tipo de pago 1: 4.1
    % con propina cero en el tipo de pago 2: 100.0
    % con propina cero en el tipo de pago 3: 100.0
    % con propina cero en el tipo de pago 4: 100.0
  viajes que no se pagan con tarjeta: 7434
    y su porcentaje de propina cero: 100.0
  viajes con tarjeta utilizables: 15203


**El importe total contiene la propina en 22.656 de 22.699 viajes.** Queda prohibida como
variable. El coste antes de propina se reconstruye restándola, y eso sí es lo que el pasajero
ve en la pantalla cuando le sale el aviso.

## 2. El efectivo, que decide el proyecto entero

Antes de modelar, el porcentaje de propina cero **por tipo de pago**.

In [3]:
by_payment = df.groupby("payment_type").agg(
    viajes=("tip_amount", "size"),
    propina_cero_pct=("tip_amount", lambda s: round(100 * s.eq(0).mean(), 1)))
print(by_payment.to_string())

              viajes  propina_cero_pct
payment_type                          
1              15265               4.1
2               7267             100.0
3                121             100.0
4                 46             100.0


**Los tres tipos que no son tarjeta están clavados en el 100 %.** No es que esa gente no deje
propina: el taxímetro solo apunta la de la tarjeta.

Entrenar con el archivo entero le enseñaría al modelo *«efectivo, luego no hay propina»*, que
es una propiedad de la máquina disfrazada de hallazgo, y saldría un modelo excelente por la
razón equivocada. Se apartan 7.434 viajes.

![La propina en efectivo no se registra](03_figures/01_efectivo.png)

## 3. Qué significa «generoso», leído de la distribución

El umbral no se elige. Se mira dónde se acumulan las propinas.

In [4]:
card = build_target(card, results)
top = (100 * card.rate).round(1).value_counts().head(6)
print("los porcentajes de propina mas repetidos")
print(top.to_string())


3. Qué es «generoso», leído de los datos y no elegido
  mediana de la propina, en porcentaje: 19.97
  viajes que dejan exactamente el 20,00 %, que es el botón por defecto: 4363
    en porcentaje: 28.7
  viajes en uno de los tres botones de la máquina: 5604
    en porcentaje: 36.9
  generoso es pasar del botón por defecto, no dejar algo: 0.2
    y sale este porcentaje de generosos: 23.11
  mediana 19.97 %, y 4363 viajes en el botón exacto del 20 %
  generosos: 23.11 %
los porcentajes de propina mas repetidos
rate
20.0    4956
19.9    2231
25.0    1170
0.0      615
30.0     424
19.8     348


**Tres torres, y son los tres botones que ofrece la máquina.** 4.363 viajes dejan exactamente
el 20,00 %, el 28,7 % de los pagos con tarjeta, y la mediana cae en 19,97 %, o sea justo en
ese botón.

Quien pulsa el botón por defecto **no decidió ser generoso: aceptó**. Por eso el umbral es
pasar del 20 %, y con él los generosos son el 23,11 %.

![Lo que se predice es qué botón pulsó](03_figures/02_el_boton.png)

## 4. El control, antes que cualquier modelo

Con un 23 % de positivos, contestar siempre «no generoso» ya saca buena nota sin hacer nada.
Esa es la barra.

In [5]:
train, validation, test = split(card, results)
print(f"contestar siempre que no: exactitud {1 - validation.generous.mean():.4f}, "
      f"sensibilidad 0.0000")


4. Partición en tres
  entrenamiento 9121   validación 3041   prueba 3041
contestar siempre que no: exactitud 0.7688, sensibilidad 0.0000


## 5. Los modelos, y la tabla que responde

Medidos en prueba una sola vez, después de elegir campeón en validación.

In [6]:
X, y = train[FEATURES], train.generous
Xt, yt = test[FEATURES], test.generous

forest = RandomForestClassifier(n_estimators=300, max_depth=5, max_features=0.5,
                                min_samples_leaf=50, random_state=SEED, n_jobs=-1).fit(X, y)
logistic = LogisticRegression(max_iter=3000).fit(X, y)

print(f"{'contestar siempre que no':26} exactitud {1-yt.mean():.4f}   "
      f"sensibilidad 0.0000   AUC 0.5000")
for name, model in [("logistica", logistic), ("bosque ajustado", forest)]:
    p = model.predict_proba(Xt)[:, 1]
    d = (p >= 0.5).astype(int)
    print(f"{name:26} exactitud {accuracy_score(yt, d):.4f}   "
          f"sensibilidad {recall_score(yt, d):.4f}   AUC {roc_auc_score(yt, p):.4f}")

contestar siempre que no   exactitud 0.7688   sensibilidad 0.0000   AUC 0.5000
logistica                  exactitud 0.7688   sensibilidad 0.0000   AUC 0.5395
bosque ajustado            exactitud 0.7688   sensibilidad 0.0000   AUC 0.6073


**Ni la logística ni el bosque señalan a nadie.** Su sensibilidad es cero, o sea que toman
exactamente las mismas decisiones que contestar siempre que no, y por eso su exactitud es
idéntica hasta el cuarto decimal.

Y aquí hay un matiz que vale la pena guardarse: **los árboles le ganan a la logística en
AUC**, 0,6073 contra 0,5395, por primera vez en todo el curso. Ordenan mejor los casos. Y
aun así no deciden nada distinto.

**Un AUC mejor no es un modelo utilizable.** El AUC mide si sabes ordenar; la sensibilidad
mide si sirves para decidir.

![Ninguno decide nada distinto](03_figures/03_no_hay_senal.png)

## 6. La importancia, y por qué mirarla aquí es una trampa

In [7]:
for name, value in sorted(zip(FEATURES, forest.feature_importances_),
                          key=lambda pair: -pair[1])[:5]:
    print(f"{name:22} {value:.4f}")

base                   0.3757
fare_amount            0.1455
duration               0.1412
trip_distance          0.0887
speed                  0.0876


Sale un reparto perfectamente presentable, con el coste antes de propina a la cabeza. **Y es
el reparto de importancia de un modelo con un AUC de 0,6073**, o sea de un modelo que no
funciona.

La importancia siempre suma 1 entre las variables que le des, acierte el modelo o no. Leerla
sin comprobar antes si el modelo sirve produce conclusiones sobre nada.

![Importancia de un modelo que no funciona](03_figures/04_importancia.png)

## 7. Qué se decide con esto

**No construir esto.** El encargo no se puede cumplir con los datos disponibles, y decirlo
con pruebas es el entregable.

**Y la palanca que sí existe:** el 28,7 % de los pasajeros deja exactamente lo que la máquina
propone. **Mover el botón por defecto mueve más dinero que cualquier modelo**, y eso se puede
probar con un experimento A/B como los del Curso 3.